# 📖 Notebook 9: Storage & Secrets — Persisting Data and Managing Configuration

Welcome back! In earlier notebooks, our FastAPI services mostly acted like stateless apps. In the real world, many workloads need configuration, passwords, and data that survives a restart. This notebook shows how Kubernetes handles those needs without baking everything into the container image.

> **Prerequisites: Notebooks 01–02.** Everything below runs in the `k8s-lab` namespace.

In [ ]:
# ── Preflight ────────────────────────────────────────────────────────────
# Every later cell shells out to these tools. Without this check a missing
# binary fails silently inside a `!` magic and you only see a confusing
# downstream error (e.g. FileNotFoundError from %%writefile) instead of
# "helm is not installed". Run this first.
import shutil
import subprocess

REQUIRED = ['kubectl']
INSTALL_HINTS = {
    'kubectl': 'https://kubernetes.io/docs/tasks/tools/  (or `brew install kubectl`)',
}

missing = [b for b in REQUIRED if shutil.which(b) is None]
if missing:
    hint = '\n'.join(f'  - {b}: {INSTALL_HINTS[b]}' for b in missing)
    raise RuntimeError(
        f"Missing required CLI tool(s): {', '.join(missing)}\n"
        f"Install them, then re-run this cell:\n{hint}"
    )

# A reachable cluster is required too -- `kubectl` alone is not enough.
probe = subprocess.run(
    ['kubectl', 'cluster-info'], capture_output=True, text=True
)
if probe.returncode != 0:
    raise RuntimeError(
        'No reachable Kubernetes cluster. Start the one from notebook 1:\n'
        '  minikube start --cpus=4 --memory=6144 --driver=docker\n'
        f'kubectl said: {probe.stderr.strip()[:300]}'
    )

print('Preflight OK:', ', '.join(REQUIRED), '+ cluster reachable')

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────
# `!kubectl ...` prints but never fails a cell, so every claim this notebook
# makes about persistence is also checked in Python.
import base64
import json
import subprocess
import time

NS = "k8s-lab"


def kget(*args, ns=NS):
    cmd = ["kubectl", "get", *args, "-o", "json"] + (["-n", ns] if ns else [])
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("kubectl failed: " + r.stderr.strip()[:400])
    return json.loads(r.stdout)


def in_pod(pod, *cmd, container=None, ns=NS):
    """Run a command inside a pod and return (rc, stdout+stderr)."""
    c = ["kubectl", "exec", "-n", ns, pod] + (["-c", container] if container else []) + ["--", *cmd]
    r = subprocess.run(c, capture_output=True, text=True, timeout=120)
    return r.returncode, (r.stdout + r.stderr).strip()


def wait_until(predicate, timeout=170, interval=3, what="condition"):
    deadline = time.time() + timeout
    while time.time() < deadline:
        value = predicate()
        if value:
            return value
        time.sleep(interval)
    raise AssertionError(f"timed out after {timeout}s waiting for {what}")


print("helpers ready")

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain why containers are **ephemeral** and why that matters
- Use **ConfigMaps** for non-secret configuration
- Use **Secrets** for sensitive values such as usernames and passwords
- Explain why Kubernetes Secrets are base64-encoded but not encrypted by default
- Create and use a **PersistentVolumeClaim (PVC)**
- Describe the relationship between **PersistentVolumes (PV)**, **PersistentVolumeClaims (PVC)**, and **StorageClasses**
- Choose the right **access mode**, and recognise a PVC that will never bind
- Explain why a StatefulSet needs a **headless** Service
- Deploy a simple **StatefulSet** with persistent storage
- Understand the **External Secrets** pattern for production systems

## 🛠️ Setup

Before you run anything:

- Make sure you already completed the earlier Kubernetes notebooks
- Start Minikube if it is not running
- Run this lab from `03-technologies/container-orchestration/kubernetes/` so the sample manifests are easy to find
- Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.

We will use the sample microservices in the `k8s-lab` namespace:

- `api-gateway` on port `8000`
- `user-service` on port `8001`
- `order-service` on port `8002`

In [ ]:
!pwd
!kubectl config current-context
!kubectl get ns k8s-lab
!kubectl get pods -n k8s-lab
!kubectl get svc -n k8s-lab

## The Problem: Containers are Ephemeral

A container is designed to be disposable. Kubernetes can stop it, move it, or recreate it at any time. That is great for reliability, but it creates a new problem: **anything stored inside the container filesystem can disappear when the pod disappears**.

Think of a pod like a hotel room. You can leave your suitcase on the bed, but if the hotel closes your room and gives you a new one, your suitcase is gone unless you stored it somewhere permanent.

### ✅ Exercise
Create a pod, write a file inside it, delete the pod, then recreate it. You will see that the file does not come back.

In [ ]:
!kubectl delete pod ephemeral-demo -n k8s-lab --ignore-not-found
!kubectl run ephemeral-demo --image=busybox:1.36 -n k8s-lab --restart=Never --command -- sh -c "mkdir -p /demo && echo 'hello from an ephemeral pod' > /demo/demo.txt && sleep 3600"
!kubectl wait --for=condition=Ready pod/ephemeral-demo -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab ephemeral-demo -- cat /demo/demo.txt

rc, first = in_pod("ephemeral-demo", "cat", "/demo/demo.txt")
assert rc == 0 and "hello from an ephemeral pod" in first, first

!kubectl delete pod ephemeral-demo -n k8s-lab
!kubectl run ephemeral-demo --image=busybox:1.36 -n k8s-lab --restart=Never --command -- sh -c "mkdir -p /demo && sleep 3600"
!kubectl wait --for=condition=Ready pod/ephemeral-demo -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab ephemeral-demo -- ls /demo

# The whole point: the file is gone. A container's writable layer is created
# fresh with the container and destroyed with it.
rc, out = in_pod("ephemeral-demo", "ls", "/demo")
assert out == "", f"expected /demo to be empty in the new pod, found: {out!r}"
rc, out = in_pod("ephemeral-demo", "cat", "/demo/demo.txt")
assert rc != 0, "demo.txt survived the pod -- it should not have"
print("\n✅ /demo is empty: the file did not survive the pod")

### ASCII Diagram: How Persistent Storage Fits Together

```
┌──────────────┐
│     Pod      │
│ writes files │
└──────┬───────┘
       │ mounts
       ▼
┌──────────────┐
│     PVC      │  PersistentVolumeClaim
│ "I need 1Gi" │
└──────┬───────┘
       │ bound to
       ▼
┌──────────────┐
│      PV      │  PersistentVolume
│ actual disk  │
└──────┬───────┘
       │ backed by
       ▼
┌──────────────────────┐
│ Physical Storage     │
│ disk / SSD / cloud   │
└──────────────────────┘
```

The PVC is the request. The PV is the actual storage. In many clusters, a StorageClass automatically creates the PV for you when you create the PVC.

## 🧾 ConfigMaps

A **ConfigMap** stores non-secret configuration as key-value pairs. This is a good place for things like log level, environment name, feature flags, and cache settings.

Why is this useful? Because your container image stays the same, while your configuration can change per environment. The same image can run in dev, staging, and production with different settings.

### ✅ Exercise
Create a ConfigMap from literal values right from the command line.

In [ ]:
# NOTE the name: `demo-config`, not `app-config`. The shared ../manifests/configmap.yaml
# creates `app-config`, and order-service pulls its whole environment from it with
# `envFrom`. Overwriting it here with different keys -- or deleting it in the cleanup
# cell -- would break order-service in every later notebook. Scratch objects get
# scratch names.
!kubectl delete configmap demo-config -n k8s-lab --ignore-not-found
!kubectl create configmap demo-config --from-literal=LOG_LEVEL=info --from-literal=ENV=dev -n k8s-lab
!kubectl get configmap demo-config -n k8s-lab -o yaml

cm = kget("configmap", "demo-config")
assert cm["data"] == {"LOG_LEVEL": "info", "ENV": "dev"}, cm["data"]
assert kget("configmap", "app-config")["data"]["LOG_LEVEL"], \
    "the shared app-config ConfigMap must still exist -- order-service consumes it"
print("\n✅ demo-config created; the shared app-config is untouched")

## Using a ConfigMap in a Pod

Kubernetes lets a pod consume ConfigMap data in two very common ways:

1. **As environment variables** — easy for apps that already read environment variables
2. **As mounted files in a volume** — useful when an app expects config files

We will do both in the same pod so you can compare the behavior.

### ✅ Exercise
Write a pod manifest that reads the same ConfigMap both as env vars and as files under `/config`.

In [ ]:
%%writefile config-demo-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: config-demo
  namespace: k8s-lab
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "sleep 3600"]
      env:
        - name: LOG_LEVEL
          valueFrom:
            configMapKeyRef:
              name: demo-config
              key: LOG_LEVEL
        - name: ENV
          valueFrom:
            configMapKeyRef:
              name: demo-config
              key: ENV
      volumeMounts:
        - name: config-volume
          mountPath: /config
  volumes:
    - name: config-volume
      configMap:
        name: demo-config

In [ ]:
!kubectl delete pod config-demo -n k8s-lab --ignore-not-found
!kubectl apply -f config-demo-pod.yaml
!kubectl wait --for=condition=Ready pod/config-demo -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab config-demo -- printenv LOG_LEVEL ENV
!kubectl exec -n k8s-lab config-demo -- sh -c "ls /config && echo '---' && cat /config/LOG_LEVEL && echo && cat /config/ENV"

# Both consumption styles should show the same values right now. They will stop
# agreeing in the very next section, which is the point of it.
rc, env = in_pod("config-demo", "printenv", "LOG_LEVEL", "ENV")
assert env.split() == ["info", "dev"], f"env vars are {env.split()}"
rc, mounted = in_pod("config-demo", "sh", "-c", "cat /config/LOG_LEVEL; echo; cat /config/ENV")
assert mounted.split() == ["info", "dev"], f"mounted files say {mounted.split()}"
print("\n✅ env vars and mounted files both read info/dev")

## Updating a ConfigMap

This is one of the most important behaviors to understand:

- **Environment variables never change in a running pod.** They are copied in at container
  start and are then just process environment. Only a new pod picks up a new value.
- **Mounted ConfigMap files do update in place** — but not instantly. The kubelet
  re-syncs its config cache on a period (default ~1 minute) plus the cache TTL, so expect
  up to ~2 minutes. The update is atomic: the kubelet writes a new directory and swaps a
  symlink, so a reader never sees a half-written file.

Two exceptions worth knowing:

- A volume mounted with `subPath` does **not** receive updates. If you mounted a single
  file with `subPath: nginx.conf`, it is frozen at pod start. This surprises people
  constantly.
- A ConfigMap marked `immutable: true` cannot be changed at all — which is a feature: it
  removes the kubelet's watch on it, materially reducing API-server load in large clusters,
  and forces changes to go through a new ConfigMap name plus a rollout.

And even when the *file* updates, **your application still has to re-read it**. Most do
not. That is why the common production pattern is the opposite of hot-reload: hash the
ConfigMap's contents into a pod annotation so that changing it forces a normal rolling
update. Deliberate, observable, rollback-able.

### ✅ Exercise
Update the ConfigMap and compare the env var with the mounted file. Then restart the pod to refresh the env var.

In [ ]:
!kubectl create configmap demo-config --from-literal=LOG_LEVEL=debug --from-literal=ENV=prod -n k8s-lab -o yaml --dry-run=client | kubectl apply -f -
!kubectl exec -n k8s-lab config-demo -- printenv LOG_LEVEL ENV
!kubectl exec -n k8s-lab config-demo -- sh -c "echo 'Mounted files may take a short moment to refresh'; ls /config; echo '---'; cat /config/LOG_LEVEL; echo; cat /config/ENV"

assert kget("configmap", "demo-config")["data"] == {"LOG_LEVEL": "debug", "ENV": "prod"}

# 1. The environment variables have NOT changed and never will: they were copied
#    into the process environment at container start.
rc, env = in_pod("config-demo", "printenv", "LOG_LEVEL", "ENV")
assert env.split() == ["info", "dev"], (
    f"env vars changed to {env.split()} in a running pod -- that is not supposed to "
    "be possible; a running process's environment cannot be rewritten from outside"
)
print("env vars in the running pod:   ", env.split(), "  <- still the OLD values")

# 2. The mounted files DO update -- eventually. The kubelet re-syncs on a period
#    (default ~1 minute) plus its cache TTL, so poll rather than checking once and
#    concluding that mounts do not update either.
print("\nwaiting for the kubelet to re-sync the mounted volume (up to ~2 minutes)...")
mounted = wait_until(
    lambda: in_pod("config-demo", "sh", "-c", "cat /config/LOG_LEVEL")[1] == "debug",
    timeout=170, interval=10, what="the mounted ConfigMap file to pick up the new value")
rc, files = in_pod("config-demo", "sh", "-c", "cat /config/LOG_LEVEL; echo; cat /config/ENV")
print("mounted files in the same pod: ", files.split(), "  <- the NEW values")
print("\n✅ same ConfigMap, same pod, two different answers: env is frozen, the mount is live")

In [ ]:
!kubectl delete pod config-demo -n k8s-lab
!kubectl apply -f config-demo-pod.yaml
!kubectl wait --for=condition=Ready pod/config-demo -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab config-demo -- printenv LOG_LEVEL ENV

# A new container start is the ONLY thing that refreshes an env var from a
# ConfigMap. In a Deployment this is `kubectl rollout restart`.
rc, env = in_pod("config-demo", "printenv", "LOG_LEVEL", "ENV")
assert env.split() == ["debug", "prod"], \
    f"the recreated pod should read the new values, got {env.split()}"
print("\n✅ a new pod picks up debug/prod -- restarting is what makes env vars change")

## 🔐 Secrets

A **Secret** is similar to a ConfigMap, but it is meant for sensitive data such as passwords, tokens, and API keys. Kubernetes treats Secrets differently in tooling and APIs, but a Secret is still just a Kubernetes object stored in the cluster.

### ✅ Exercise
Create a Secret for database credentials.

In [ ]:
!kubectl delete secret db-creds -n k8s-lab --ignore-not-found
!kubectl create secret generic db-creds --from-literal=DB_USER=admin --from-literal=DB_PASS=secret123 -n k8s-lab
!kubectl get secret db-creds -n k8s-lab -o yaml

## Base64 Encoding vs Encryption

This is a very common beginner misunderstanding, so let's make it simple:

- **Base64 encoding** changes the format of data so it is safe to move around as text
- **Encryption** scrambles the data so only someone with the right key can read it

Base64 is like writing a message in a different alphabet that anyone can translate back.
Encryption is like locking the message in a safe.

> **By default, Kubernetes Secrets are base64-encoded and stored in etcd in plaintext.
> They are NOT encrypted at rest.** Base64 is an encoding, not a cipher — there is no key
> involved, and reversing it is a single command with no credentials.

What that actually means in practice:

- Anyone with `get secrets` RBAC in the namespace can read every value. Note that the
  built-in `view` and `edit` ClusterRoles deliberately exclude `secrets` for this reason —
  but `admin` and `cluster-admin` do not.
- Anyone who can read the etcd data files or an etcd backup can read every Secret in the
  cluster. Backups are the underrated part: an unencrypted etcd snapshot on someone's
  laptop is your entire secret store.
- Mounting a Secret as an **environment variable** leaks it further than mounting it as a
  file: env vars show up in `kubectl describe pod` output for some controllers, in crash
  dumps, and in child processes. Prefer a file mount for anything sensitive.

What to actually do about it, in increasing order of strength:

1. **RBAC** — restrict `get`/`list` on secrets to the ServiceAccounts that need them.
2. **Encryption at rest** — the cluster admin adds an `EncryptionConfiguration` to the API
   server so etcd stores ciphertext. This is *not* on by default in most distributions,
   and it must be enabled deliberately.
3. **A KMS provider** — the encryption key itself lives in a cloud HSM, not on the API
   server's disk.
4. **Do not store the secret in Kubernetes at all** — External Secrets Operator or the
   Secrets Store CSI driver pull from Vault / AWS / Azure / GCP at pod start. That is the
   pattern at the end of this notebook.

### ✅ Exercise
Decode the stored values and prove to yourself that base64 is reversible.

In [ ]:
# One command, no key, no credentials. That is the whole point.
!kubectl get secret db-creds -n k8s-lab -o jsonpath='{.data.DB_USER}' | base64 --decode && echo
!kubectl get secret db-creds -n k8s-lab -o jsonpath='{.data.DB_PASS}' | base64 --decode && echo

print()
# Newer kubectl can do it for you, which makes the point even more bluntly:
!kubectl get secret db-creds -n k8s-lab -o jsonpath='{.data}' && echo

print()
# Is encryption at rest even enabled on this cluster? On minikube: no.
!kubectl -n kube-system get pod -l component=kube-apiserver -o jsonpath='{.items[0].spec.containers[0].command}' 2>/dev/null | tr ',' '\n' | grep -i encryption || echo "(no --encryption-provider-config flag: etcd stores these values in plaintext)"

# The claim is "base64 is an encoding, not a cipher". Demonstrate it: recover the
# password from the stored bytes with nothing but a decoder.
stored = kget("secret", "db-creds")["data"]
recovered = {k: base64.b64decode(v).decode() for k, v in stored.items()}
assert recovered == {"DB_USER": "admin", "DB_PASS": "secret123"}, recovered
print(f"\nrecovered from the stored bytes with no key at all: {recovered}")

# And confirm this API server really has no encryption provider configured, so
# the same bytes are what sits in etcd and in any etcd backup.
cmd = kget("pod", "-l", "component=kube-apiserver",
           ns="kube-system")["items"][0]["spec"]["containers"][0]["command"]
encryption_flags = [f for f in cmd if "encryption" in f]
assert not encryption_flags, \
    f"this cluster DOES have encryption at rest configured: {encryption_flags}"
print("✅ no --encryption-provider-config on this API server: etcd holds these in the clear")

## Mounting Secrets as Environment Variables

Just like ConfigMaps, Secrets can be injected into a pod. One very common pattern is to expose secret values as environment variables that the app reads on startup.

### ✅ Exercise
Create a pod that reads `DB_USER` and `DB_PASS` from the Secret.

In [ ]:
%%writefile secret-demo-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: secret-demo
  namespace: k8s-lab
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "sleep 3600"]
      env:
        - name: DB_USER
          valueFrom:
            secretKeyRef:
              name: db-creds
              key: DB_USER
        - name: DB_PASS
          valueFrom:
            secretKeyRef:
              name: db-creds
              key: DB_PASS

In [ ]:
!kubectl delete pod secret-demo -n k8s-lab --ignore-not-found
!kubectl apply -f secret-demo-pod.yaml
!kubectl wait --for=condition=Ready pod/secret-demo -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab secret-demo -- printenv DB_USER DB_PASS

rc, env = in_pod("secret-demo", "printenv", "DB_USER", "DB_PASS")
assert env.split() == ["admin", "secret123"], f"secret env vars are {env.split()}"

# Worth seeing rather than just being told: the value is now visible in the pod
# spec's env definition too, which is why a file mount leaks less than an env var.
print("\n✅ the Secret arrived as plain environment variables inside the container")

## 💾 PersistentVolumes and PersistentVolumeClaims

Now we solve the big storage problem. Instead of writing data only inside the pod filesystem, we attach storage through a **PersistentVolumeClaim**.

The split is deliberate: a **PVC** is what the *application author* writes ("I need 1Gi of
ReadWriteOnce storage"), and a **PV** is the actual piece of storage. Keeping them
separate is what lets the same Deployment YAML run on minikube's hostPath and on an AWS
EBS volume without changing a line.

### Static vs dynamic provisioning

- **Static**: a cluster admin pre-creates PVs. A new PVC is matched against existing
  unbound PVs by capacity, access mode, and StorageClass. If nothing matches, the PVC sits
  in `Pending` **forever** — there is no error, no timeout, no event that says "give up".
- **Dynamic**: the PVC names a StorageClass, and its provisioner creates a PV on demand.
  This is what almost every real cluster does, and what minikube's default `standard`
  class does.

Binding is **one-to-one and exclusive**: once a PVC binds to a PV, no other PVC can use
that PV, even if there is space left.

### Access modes

`accessModes` on a PVC is a **request**, matched against what the storage backend can
actually do. It is not enforced isolation — asking for `ReadWriteMany` from a backend that
cannot do it gets you a `Pending` PVC, not an error.

| Mode | Short | Meaning |
|---|---|---|
| `ReadWriteOnce` | RWO | Mounted read-write by **one node**. Multiple pods may share it *if they are on the same node*. This is the default and what block storage (EBS, GCE PD, most CSI drivers) supports. |
| `ReadOnlyMany` | ROX | Mounted read-only by many nodes. |
| `ReadWriteMany` | RWX | Mounted read-write by many nodes. Needs a **file** backend (NFS, EFS, CephFS, Azure Files). Block storage cannot do this — the single most common cause of a stuck PVC. |
| `ReadWriteOncePod` | RWOP | Exactly **one pod** cluster-wide. Kubernetes 1.29+. Use it for a single-writer database, where RWO's "same node" loophole is not strict enough. |

Note the RWO subtlety: "Once" means once per **node**, not once per **pod**. Two pods of
the same Deployment scheduled to the same node can both mount an RWO volume read-write and
corrupt each other's data. `ReadWriteOncePod` exists precisely because people kept
assuming otherwise.

### ✅ Exercise
Inspect the storage classes, then create a PVC for 1Gi of storage.

In [ ]:
# The class marked (default) is what a PVC with no `storageClassName` gets.
!kubectl get storageclass

In [ ]:
%%writefile storage-demo-pvc.yaml
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: storage-demo-pvc
  namespace: k8s-lab
spec:
  accessModes:
    - ReadWriteOnce
  resources:
    requests:
      storage: 1Gi

In [ ]:
!kubectl apply -f storage-demo-pvc.yaml
!kubectl get pvc storage-demo-pvc -n k8s-lab
!kubectl get pv

pvc = wait_until(
    lambda: (lambda c: c if c["status"]["phase"] == "Bound" else None)(
        kget("pvc", "storage-demo-pvc")),
    timeout=120, interval=3, what="the PVC to bind")
pv = kget("pv", pvc["spec"]["volumeName"], ns=None)
print(f"\n✅ bound to PV {pv['metadata']['name']} "
      f"({pv['spec']['capacity']['storage']}, class {pv['spec']['storageClassName']}, "
      f"reclaim {pv['spec']['persistentVolumeReclaimPolicy']})")
assert pv["spec"]["claimRef"]["name"] == "storage-demo-pvc", \
    "binding is one-to-one: the PV should point back at exactly this claim"

### 💥 Failure First: a PVC That Never Binds

The PVC above went straight to `Bound` because minikube has a default StorageClass that
provisions on demand. Now ask for something nothing can satisfy, and watch how quiet the
failure is.

**Choosing a request that actually fails takes a little care on minikube**, and the reason
is itself worth knowing. The obvious candidate — `accessModes: [ReadWriteMany]` with
`storage: 500Gi` — does *not* fail here. minikube's default provisioner is
`k8s.io/minikube-hostpath`, which creates a directory on the node: it ignores the
requested capacity (you get the node's whole disk, not 500Gi) and it accepts every access
mode, including `ReadWriteMany`, which it cannot honour in any meaningful sense. Ask a
real EBS-backed StorageClass for RWX and the provisioner refuses; ask hostPath and it
cheerfully says yes.

So to see the failure we take the provisioner out of the picture, which is the **static
provisioning** case from the table above: `storageClassName: ""` means "do not
dynamically provision anything, bind me to a pre-created PV that matches". No such PV
exists, so the claim waits — forever, with no error and no timeout.

That is not a contrived scenario. It is what happens the first time a manifest written
for a cluster with static PVs, or one that names a StorageClass this cluster does not
have, is applied somewhere new.

In [ ]:
%%writefile stuck-pvc.yaml
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: stuck-pvc
  namespace: k8s-lab
spec:
  # storageClassName: "" (the empty string, not "omitted") disables dynamic
  # provisioning for this claim. Kubernetes will only bind it to an existing,
  # unbound PV whose capacity and access modes match. There is no such PV here,
  # so nothing happens -- no provisioner is even asked.
  storageClassName: ""
  accessModes:
    - ReadWriteMany
  resources:
    requests:
      storage: 500Gi

In [ ]:
!kubectl apply -f stuck-pvc.yaml

import time
time.sleep(20)

print("\n--- STATUS ---")
!kubectl get pvc stuck-pvc -n k8s-lab

print("\n--- the only place the reason appears ---")
!kubectl describe pvc stuck-pvc -n k8s-lab | tail -n 12

# "Pending, indefinitely, with no error" is the claim. Check all three parts:
# the phase is Pending, no PV was bound, and the object carries no failure
# condition that an automated check could alert on.
stuck = kget("pvc", "stuck-pvc")
assert stuck["status"]["phase"] == "Pending", \
    f"the unsatisfiable PVC bound anyway: {stuck['status']}"
assert "volumeName" not in stuck["spec"], "nothing should have bound to it"
assert not stuck["status"].get("conditions"), \
    f"there is no failure condition to alert on -- that is the point: {stuck['status']}"
print("\n✅ Pending, no volume, and NO status condition anyone could alert on")

In [ ]:
# The PVC itself is not the symptom people see. This is:
!kubectl delete pod stuck-consumer -n k8s-lab --ignore-not-found
!kubectl run stuck-consumer -n k8s-lab --image=busybox:1.36 --restart=Never \
    --overrides='{"spec":{"containers":[{"name":"app","image":"busybox:1.36","command":["sleep","3600"],"volumeMounts":[{"name":"d","mountPath":"/data"}]}],"volumes":[{"name":"d","persistentVolumeClaim":{"claimName":"stuck-pvc"}}]}}'

time.sleep(20)
!kubectl get pod stuck-consumer -n k8s-lab
!kubectl describe pod stuck-consumer -n k8s-lab | grep -A5 Events

# The pod is Pending too, and the scheduler's message points at the PVC -- but
# only if you read it. People usually stare at the pod, the node and the
# scheduler for a while before thinking to run `kubectl describe pvc`.
pod = kget("pod", "stuck-consumer")
assert pod["status"]["phase"] == "Pending", \
    f"a pod mounting an unbound PVC cannot be scheduled; this one is {pod['status']['phase']}"
reason = " ".join(c.get("message", "") for c in pod["status"].get("conditions", []))
assert "unbound immediate PersistentVolumeClaims" in reason or "PersistentVolumeClaim" in reason, \
    f"expected the scheduler to blame the PVC, got: {reason}"
print(f"\n✅ the pod is Pending too: {reason.strip()[:140]}")

!kubectl delete pod stuck-consumer -n k8s-lab --ignore-not-found --wait=false

`Pending`, indefinitely. No error, no timeout, no failed condition — a PVC waits forever
for a PV that may never exist.

The downstream symptom is worse than the PVC itself, and you just saw it: any pod that
mounts the claim stays `Pending` too, with
`0/1 nodes are available: pod has unbound immediate PersistentVolumeClaims`. People
usually go looking at the pod, the node and the scheduler before they think to run
`kubectl describe pvc`.

There is one more twist. Many cloud StorageClasses set
`volumeBindingMode: WaitForFirstConsumer` instead of `Immediate`. That makes a *healthy*
PVC also sit in `Pending` until a pod that uses it is scheduled — because the provisioner
needs to know which availability zone to create the disk in. So `Pending` is not
automatically a problem: check `kubectl get storageclass -o wide` for the binding mode
before you go debugging.

In [ ]:
!kubectl delete -f stuck-pvc.yaml --ignore-not-found
!rm -f stuck-pvc.yaml

## Proving That Data Persists

A PVC only becomes meaningful when a pod mounts it. We will create a pod that writes data into `/data`, delete the pod, recreate it, and then confirm that the file is still there.

### ✅ Exercise
Mount the PVC in a pod and test data persistence.

In [ ]:
%%writefile pvc-writer-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: pvc-writer
  namespace: k8s-lab
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "echo 'pod started' >> /data/history.txt && sleep 3600"]
      volumeMounts:
        - name: app-data
          mountPath: /data
  volumes:
    - name: app-data
      persistentVolumeClaim:
        claimName: storage-demo-pvc

In [ ]:
!kubectl delete pod pvc-writer -n k8s-lab --ignore-not-found
!kubectl apply -f pvc-writer-pod.yaml
!kubectl wait --for=condition=Ready pod/pvc-writer -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab pvc-writer -- sh -c "echo 'this line survives pod deletion' >> /data/history.txt && cat /data/history.txt"

rc, before = in_pod("pvc-writer", "cat", "/data/history.txt")
assert "this line survives pod deletion" in before, before

!kubectl delete pod pvc-writer -n k8s-lab
!kubectl apply -f pvc-writer-pod.yaml
!kubectl wait --for=condition=Ready pod/pvc-writer -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab pvc-writer -- cat /data/history.txt

# The opposite of the ephemeral demo at the top: a completely new pod, and the
# file is still there. Note the extra "pod started" line -- the new container's
# command appended to the SAME file.
rc, after = in_pod("pvc-writer", "cat", "/data/history.txt")
assert "this line survives pod deletion" in after, \
    f"the data did not survive the pod: {after!r}"
assert after.count("pod started") == before.count("pod started") + 1, \
    f"expected the new pod to append one more line:\nbefore={before!r}\nafter={after!r}"
print(f"\n✅ new pod, same PVC, {len(after.splitlines())} lines of history intact")

## StorageClass Basics

A **StorageClass** tells Kubernetes *how* to provision storage. Think of it like a template for disk creation.

Important ideas:

- **Provisioner**: the plugin or driver that actually creates storage
- **Reclaim policy** (`persistentVolumeReclaimPolicy`): what happens to the PV *and the
  real disk* when the PVC is deleted
  - `Delete` — the PV and the underlying storage are destroyed. This is the default for
    dynamically provisioned volumes, which means **`kubectl delete pvc` on a database is
    an irreversible data-loss command**.
  - `Retain` — the PV survives in `Released` state with the data intact, but it will not
    be reused automatically: an admin must clear `spec.claimRef` before another PVC can
    bind to it.
- **Binding mode** (`volumeBindingMode`): when the PV is actually provisioned
  - `Immediate` — as soon as the PVC is created (minikube's default)
  - `WaitForFirstConsumer` — not until a pod that uses the PVC is scheduled, so the volume
    lands in the same zone/node as the pod. Standard on cloud providers.
- **`allowVolumeExpansion`**: whether you can grow a PVC by editing
  `spec.resources.requests.storage`. Growing is often supported; **shrinking never is**.

### ✅ Exercise
Inspect the default StorageClass and look for its provisioner and reclaim policy.

In [ ]:
!kubectl get storageclass -o wide
!kubectl describe storageclass $(kubectl get storageclass -o jsonpath='{.items[0].metadata.name}')

## 🧠 StatefulSet: When Pod Identity Matters

A Deployment is great for stateless apps because any replica can replace any other replica. But databases and queues often need a stable identity and stable storage. That is where **StatefulSet** comes in.

A StatefulSet gives pods predictable names like `redis-0`, `redis-1`, and `redis-2`.
Those names are stable across restarts and rescheduling, and each pod keeps **its own**
PersistentVolumeClaim — `redis-data-redis-0` follows `redis-0` wherever it goes.

Three guarantees a Deployment does not give you:

1. **Stable network identity.** Each pod gets its own DNS name,
   `redis-0.redis.k8s-lab.svc.cluster.local`, which is why `serviceName` in the spec
   points at a **headless** Service.
2. **Stable storage.** `volumeClaimTemplates` creates one PVC per pod, and re-attaches the
   same one when the pod is recreated. Deleting the StatefulSet does **not** delete those
   PVCs — deliberately, so an accidental delete does not destroy your data. You have to
   delete them yourself, which the cleanup cell at the end does.
3. **Ordered operations.** Pods are created `0, 1, 2` and terminated `2, 1, 0`, each step
   waiting for the previous one to be Ready. Essential for quorum systems.

### Why the Service has `clusterIP: None`

The Service below is **headless**. A normal ClusterIP Service would hand out one virtual
IP and load-balance across pods — exactly wrong here, because a client that wants
`redis-0` must be able to reach *that* pod. With `clusterIP: None`, Kubernetes allocates
no virtual IP and programs no proxy rules; DNS for `redis` returns the A records of all
ready pods, and DNS for `redis-0.redis` returns that one pod's IP.

| | ClusterIP | Headless (`clusterIP: None`) |
|---|---|---|
| Virtual IP | yes | none |
| DNS for the Service name | the virtual IP | one A record per ready pod |
| Per-pod DNS name | no | yes, with a StatefulSet |
| Load balancing | kube-proxy | the client decides |

### ✅ Exercise
Deploy a tiny Redis StatefulSet with persistent storage, set a key, restart the pod, and confirm the value still exists.

In [ ]:
%%writefile redis-statefulset.yaml
apiVersion: v1
kind: Service
metadata:
  name: redis
  namespace: k8s-lab
spec:
  # Headless: no virtual IP, no kube-proxy rules. DNS returns per-pod A records,
  # and each StatefulSet pod also gets redis-<n>.redis.k8s-lab.svc.cluster.local
  clusterIP: None
  selector:
    app: redis
  ports:
    - name: redis
      port: 6379
      targetPort: 6379
---
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: redis
  namespace: k8s-lab
spec:
  serviceName: redis
  replicas: 1
  selector:
    matchLabels:
      app: redis
  template:
    metadata:
      labels:
        app: redis
    spec:
      containers:
        - name: redis
          image: redis:7-alpine
          command: ["redis-server", "--appendonly", "yes"]
          ports:
            - containerPort: 6379
          volumeMounts:
            - name: redis-data
              mountPath: /data
  volumeClaimTemplates:
    - metadata:
        name: redis-data
      spec:
        accessModes: ["ReadWriteOnce"]
        resources:
          requests:
            storage: 1Gi

In [ ]:
!kubectl apply -f redis-statefulset.yaml
!kubectl rollout status statefulset/redis -n k8s-lab --timeout=120s
!kubectl get pods,pvc -n k8s-lab | grep redis

# The headless Service is what gives redis-0 its own DNS name. A ClusterIP here
# would hand out one virtual IP and load-balance, which is exactly wrong for a
# workload whose whole point is per-pod identity.
svc = kget("svc", "redis")
assert svc["spec"]["clusterIP"] == "None", \
    f"the Service must be headless for per-pod DNS, got clusterIP={svc['spec']['clusterIP']}"

pvc_before = kget("pvc", "redis-data-redis-0")
assert pvc_before["status"]["phase"] == "Bound", pvc_before["status"]
pod_before = kget("pod", "redis-0")["metadata"]["uid"]

print("\n--- write a value ---")
!kubectl exec -n k8s-lab redis-0 -- redis-cli SET notebook9 stored
!kubectl exec -n k8s-lab redis-0 -- redis-cli GET notebook9

print("\n--- destroy the pod ---")
!kubectl delete pod redis-0 -n k8s-lab
!kubectl wait --for=condition=Ready pod/redis-0 -n k8s-lab --timeout=120s

print("\n--- same name, same PVC, same data ---")
!kubectl exec -n k8s-lab redis-0 -- redis-cli GET notebook9
!kubectl get pvc -n k8s-lab -o custom-columns=NAME:.metadata.name,STATUS:.status.phase,VOLUME:.spec.volumeName | grep -E 'NAME|redis'

# All three StatefulSet guarantees, checked: same NAME, a genuinely NEW pod, the
# SAME underlying volume, and the data still there.
pod_after = kget("pod", "redis-0")["metadata"]["uid"]
assert pod_after != pod_before, "the pod was not actually replaced"
pvc_after = kget("pvc", "redis-data-redis-0")
assert pvc_after["spec"]["volumeName"] == pvc_before["spec"]["volumeName"], \
    "the replacement pod got a different volume -- identity did not follow the pod"
rc, value = in_pod("redis-0", "redis-cli", "GET", "notebook9")
assert value == "stored", f"the value did not survive the pod: {value!r}"
print(f"\n✅ redis-0 is a new pod ({pod_after[:8]}...) on the same volume "
      f"({pvc_after['spec']['volumeName'][:20]}...) with its data intact")

## 🌐 External Secrets Pattern

In production, teams often avoid storing sensitive values directly in Kubernetes. Instead, they use an external secret store such as HashiCorp Vault, AWS Secrets Manager, Azure Key Vault, or Google Secret Manager.

The flow usually looks like this:

```
┌────────────────────┐
│ ExternalSecret CRD │
└─────────┬──────────┘
          │ read rule
          ▼
┌────────────────────┐
│ External provider  │  Vault / AWS / Azure / GCP
└─────────┬──────────┘
          │ sync value
          ▼
┌────────────────────┐
│ Kubernetes Secret  │
└─────────┬──────────┘
          │ consumed by
          ▼
┌────────────────────┐
│ Pod / Deployment   │
└────────────────────┘
```

This notebook does **not** require a Vault install. We will just look at what the CRD usually looks like.

### ✅ Exercise
Write an example `ExternalSecret` manifest and read through the fields.

In [ ]:
%%writefile external-secret-example.yaml
# External Secrets Operator serves `external-secrets.io/v1` on recent releases
# (v1beta1 is still served for compatibility). Check what your cluster has with
#   kubectl api-resources --api-group=external-secrets.io
apiVersion: external-secrets.io/v1beta1
kind: ExternalSecret
metadata:
  name: db-creds-sync
  namespace: k8s-lab
spec:
  refreshInterval: 1h
  secretStoreRef:
    name: lab-secret-store
    kind: SecretStore
  target:
    name: db-creds
  data:
    - secretKey: DB_USER
      remoteRef:
        key: /k8s-lab/database
        property: username
    - secretKey: DB_PASS
      remoteRef:
        key: /k8s-lab/database
        property: password

In [ ]:
!cat external-secret-example.yaml

## ✅ Best Practices

Here are the habits you want in real projects:

- Do **not** commit real secrets to Git
- Use **ConfigMaps** for non-sensitive configuration only
- Use **Secrets** for sensitive values, but remember they need RBAC and platform protection too
- Enable **encryption at rest** for cluster data if your platform supports it
- Prefer **external secret operators** in production
- Choose the right **StorageClass** for the workload
- Use **StatefulSets** when identity and storage must stay attached to a workload

## 🧹 Clean Up

Run this when you want to remove the resources created by this notebook.

In [ ]:
!kubectl delete pod ephemeral-demo config-demo secret-demo pvc-writer stuck-consumer -n k8s-lab --ignore-not-found
!kubectl delete secret db-creds -n k8s-lab --ignore-not-found
!kubectl delete configmap demo-config -n k8s-lab --ignore-not-found
!kubectl delete pvc storage-demo-pvc stuck-pvc -n k8s-lab --ignore-not-found
!kubectl delete -f redis-statefulset.yaml --ignore-not-found

# Deleting a StatefulSet does NOT delete the PVCs its volumeClaimTemplates created.
# That is a safety feature, and it is also how clusters quietly accumulate orphaned
# disks that nobody is paying attention to. Delete them by name.
!kubectl delete pvc redis-data-redis-0 -n k8s-lab --ignore-not-found

print()
!kubectl get pvc -n k8s-lab

# Deleting a PVC whose StorageClass has reclaimPolicy: Delete destroys the
# underlying volume too. Confirm the namespace is actually empty of them, since
# orphaned PVCs are how a cluster quietly accumulates paid-for disks.
mine = {"storage-demo-pvc", "stuck-pvc", "redis-data-redis-0"}
leftover = sorted(mine & {c["metadata"]["name"] for c in kget("pvc")["items"]})
assert leftover == [], f"PVCs this notebook created are still present: {leftover}"
print("this notebook's PVCs are gone")

!rm -f config-demo-pod.yaml secret-demo-pod.yaml storage-demo-pvc.yaml pvc-writer-pod.yaml redis-statefulset.yaml external-secret-example.yaml stuck-pvc.yaml

## 🎓 What You Learned

Nice work. In this notebook you learned how Kubernetes handles the two things stateless demos usually skip: **configuration** and **data persistence**.

You practiced how to:

- Store non-secret settings in a **ConfigMap**
- Store sensitive values in a **Secret**
- Explain why base64 is not the same as encryption
- Request durable storage with a **PVC**
- See how a **StatefulSet** keeps identity and storage aligned
- Recognise a PVC stuck in `Pending`, and tell an unsatisfiable request apart from a
  perfectly healthy `WaitForFirstConsumer` binding mode
- Remember that a dynamically provisioned PV defaults to `reclaimPolicy: Delete`, so
  deleting a PVC destroys the data
- Remember that deleting a StatefulSet leaves its PVCs behind on purpose
- Understand why production teams often use **external secret managers**

In the final notebook, we will zoom out and look at production-ready platform patterns: autoscaling, multi-tenancy, disruption budgets, and backup strategy.